# 07 - Joining and Combining DataFrames

Joins add columns from related datasets. Unions append rows from compatible datasets (same schema).

## Learning objectives

By the end of this notebook, you will be able to:

- use inner and left joins deliberately;
- avoid ambiguous columns with aliases;
- anticipate how join cardinality affects row counts;
- find unmatched keys with a left-anti join; and
- append compatible rows with `unionByName`.

## Prerequisite recap

A join usually causes a shuffle so matching keys can meet. Check key quality and row counts before and after joining.

In [3]:
from pyspark.sql import functions as F

orders = spark.createDataFrame(
    [
        (1001, 'C001', 'P001', 2),
        (1002, 'C002', 'P002', 1),
        (1003, 'C001', 'P003', 3),
        (1004, 'C003', 'P001', 5),
        (1005, 'C004', 'P004', 2),
        (1006, 'C999', 'P003', 4),
    ],
    'order_id INT, customer_id STRING, product_id STRING, quantity INT',
)

customers = spark.createDataFrame(
    [
        ('C001', 'Northwind Supplies', 'North'),
        ('C002', 'Contoso Retail', 'West'),
        ('C003', 'Adventure Works', 'North'),
        ('C004', 'Fabrikam Stores', 'South'),
        ('C005', 'Wide World Importers', 'East'),
    ],
    'customer_id STRING, customer_name STRING, region STRING',
)

products = spark.createDataFrame(
    [
        ('P001', 'Notebook', 'Stationery'),
        ('P002', 'Office Chair', 'Furniture'),
        ('P003', 'Pen Set', 'Stationery'),
        ('P004', 'Desk Lamp', 'Electronics'),
    ],
    'product_id STRING, product_name STRING, category STRING',
)

account_managers = spark.createDataFrame(
    [('North', 'Maya Patel'), ('West', 'Louis Bernard')],
    'region STRING, account_manager STRING',
)

## Inner join

An inner join keeps only rows whose key appears on both sides. The order for unknown customer C999 is excluded.

In [ ]:
orders.show()
customers.show()
products.show()

In [ ]:
matched_orders = orders.join(customers, on='customer_id', how='inner')
complete_orders = matched_orders.join(products, on='product_id', how='inner')
complete_orders.select(
    'order_id', 'customer_name', 'region', 'product_name', 'category', 'quantity'
).show()

## Left join and column aliases

A left join preserves every row from the left DataFrame. Aliases make the source of same-named columns explicit and avoid ambiguous-column errors.

In [ ]:
# o = orders.alias('o')
# c = customers.alias('c')

orders_with_customers = orders.alias('o').join(
    customers.alias('c'),
    F.col('o.customer_id') == F.col('c.customer_id'),
    how='left',
).select(
    F.col('o.order_id'),
    F.col('o.customer_id'),
    F.col('c.customer_name'),
    F.col('c.region'),
)

orders_with_customers.show()

## Cardinality changes row counts

Customer C001 has two orders. Joining customers to orders therefore produces two C001 rows. A larger joined row count is not automatically an error, but it must match the relationship you expect.

In [ ]:
customer_orders = customers.join(orders, on='customer_id', how='left')
print('Customer rows:', customers.count())
print('Joined rows:', customer_orders.count())
customer_orders.orderBy('customer_id', 'order_id').show()

## Join-type reference

| Join | Rows retained |
|---|---|
| `inner` | Matching rows from both sides |
| `left` | Every left row, with nulls when unmatched |
| `right` | Every right row, with nulls when unmatched |
| `full` | Every row from both sides |
| `left_semi` | Left rows that have a match; no right columns |
| `left_anti` | Left rows that do not have a match |

In [ ]:
customers_without_orders = customers.join(
    orders.select('customer_id').distinct(),
    on='customer_id',
    how='left_anti',
)
customers_without_orders.show()

## Append rows with `unionByName`

A join combines columns using a relationship. A union appends rows. `unionByName` matches columns by name even when their order differs.

In [ ]:
online_orders = orders.filter(F.col('order_id') <= 1003).select(
    'order_id', 'customer_id', 'product_id'
)
store_orders = orders.filter(F.col('order_id') > 1003).select(
    'product_id', 'order_id', 'customer_id'
)

combined_orders = online_orders.unionByName(store_orders)
combined_orders.orderBy('order_id').show()

## Your turn

Create `order_assignments` containing `order_id`, `customer_name`, `region`, and `account_manager`. Join orders to customers and then account managers. Preserve every order, including unknown customers and regions without a manager.

In [ ]:
# Write your solution here.

### Expected result

All six order IDs remain. Orders 1001, 1003, and 1004 are assigned to Maya Patel; order 1002 to Louis Bernard; orders 1005 and 1006 have a null account manager.

### Solution - reveal after attempting

In [ ]:
o = orders.alias('o')
c = customers.alias('c')
m = account_managers.alias('m')

order_assignments = (
    o.join(c, F.col('o.customer_id') == F.col('c.customer_id'), how='left')
    .join(m, F.col('c.region') == F.col('m.region'), how='left')
    .select(
        F.col('o.order_id'),
        F.col('c.customer_name'),
        F.col('c.region'),
        F.col('m.account_manager'),
    )
    .orderBy('order_id')
)
order_assignments.show()

## Key takeaway

Choose join types from the required row-preservation rule, qualify ambiguous columns, and validate key uniqueness and row counts.

**Next:** work safely with dates and timestamps.